# Secure Communication

This exercise deals with secure communication using TLS-based examples.

## Preparation

For shell command calls, the Python module ``subprocess`` is used (https://docs.python.org/3/library/subprocess.html).

Run the following code block to provide the functions ``execute_vm_command()`` and ``execute_stm32_command()`` within this notebook.

In [1]:
import subprocess

def execute_vm_command(cmd):
    run = subprocess.run(cmd, universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)
    
def execute_stm32_command(cmd):
    ssh = 'sshpass -p "root" ssh root@192.168.13.13 '
    run = subprocess.run((ssh + cmd), universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)

Of course, all commands given in this tutorial can also be executed directly on within a Linux console.

If you want to open a terminal window, execute the next code block.

In [ ]:
execute_vm_command('gnome-terminal')

---

## Exercise 7.1: Basic TLS Server-Client Example

A big obstacle for secure device communication based on TLS is the necessity of X.509 certificates.

Within the following example, a secure machine-to-machine communication with mutual authentication based on TLS v1.3 should be established. 

Therefore, X.509 certificates for server and client are created that can be used for authentication in the TLS handshake.

### Create Server and Client Certificates

With the following commands, create a private key and a self-signed X.509 certificate for the server and the client. These can be used for TLS authentication later.

However, someone must have forgotten to set an important parameter!

Which parameter is it?

What would be a reasonable value for that parameter?

**Important Missing Parameter:**

Time: Days 

**Reasonable Value:**

365

In [4]:
execute_vm_command('openssl req -new -newkey rsa:2048 -days 365 -nodes -x509 -keyout server.key -out server.crt -config configs/server.conf')

Generating a RSA private key
................+++++
................................................................................................................+++++
writing new private key to 'server.key'
-----



In [5]:
execute_vm_command('openssl req -new -newkey rsa:2048 -days 365 -nodes -x509 -keyout client.key -out client.crt -config configs/client.conf')

Generating a RSA private key
.+++++
..............+++++
writing new private key to 'client.key'
-----



### Server Certificate Information

The following command yields detailed information about the created server certificate:

In [6]:
execute_vm_command('openssl x509 -in server.crt -text -noout')

Certificate:
    Data:
        Version: 1 (0x0)
        Serial Number:
            3c:c6:e6:cb:2e:59:7e:51:df:16:ef:2f:7f:9f:14:0a:51:c4:05:35
        Signature Algorithm: sha256WithRSAEncryption
        Issuer: C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = VM-Server
        Validity
            Not Before: Nov 26 11:06:30 2026 GMT
            Not After : Nov 26 11:06:30 2027 GMT
        Subject: C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = VM-Server
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                RSA Public-Key: (2048 bit)
                Modulus:
                    00:a8:4f:0f:d2:1d:bc:cf:e4:e6:24:59:09:73:95:
                    59:0a:39:b6:a1:f9:37:22:c2:f4:ba:1f:48:a1:24:
                    ed:77:94:0d:6b:ec:0a:41:a7:1b:f7:be:fc:b2:25:
                    b1:05:e3:02:11:b4:b0:92:40:34:1c:85:a0:cb:17:
                    c5:c8:d9:35:4d:5c:27:41:08:78:60:40:0e:39:fa:
                    76:1e:7a:cb:c8:b6:98:4d:b7:da:9f:df

### Client Certificate Information

The following command yields detailed information about the created client certificate:

In [7]:
execute_vm_command('openssl x509 -in client.crt -text -noout')

Certificate:
    Data:
        Version: 1 (0x0)
        Serial Number:
            1c:d5:ff:7f:7d:a7:d1:e7:c8:fb:ed:98:f2:8f:9e:24:61:1d:2f:2c
        Signature Algorithm: sha256WithRSAEncryption
        Issuer: C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = STM32MP1
        Validity
            Not Before: Nov 26 11:06:47 2026 GMT
            Not After : Nov 26 11:06:47 2027 GMT
        Subject: C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = STM32MP1
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                RSA Public-Key: (2048 bit)
                Modulus:
                    00:a9:16:56:c4:68:a4:18:15:23:10:d9:73:3d:ad:
                    95:ac:a3:05:1a:3a:fa:2c:15:94:9d:da:55:23:c4:
                    40:6a:d8:77:a5:c5:c6:2d:69:dd:73:f3:39:a5:cb:
                    3b:9c:ed:16:5e:52:ac:08:15:77:2c:77:4f:64:30:
                    50:3b:48:7f:be:37:4a:cc:f1:bd:e0:b8:27:e4:8f:
                    b5:c1:22:26:c3:d1:0d:50:c3:70:db:c7:3

What do ``CN`` and ``O`` stand for and which values do they have in the two certificates?

**Meaning of ``CN``:**

Common Name

**Meaning of ``O``:**

Organization

**Values of ``CN`` and ``O`` for Server and Client:**

Server:
* ``CN``: VM-Server
* ``O``: HSA_innos

Client:
* ``CN``: STM32MP1
* ``O``: HSA_innos

### Copy Client Data

Create the directory ``/home/root/secure_communication`` on your STM32MP1 board for this exercise.

In [8]:
execute_stm32_command('mkdir -p /home/root/secure_communication')

Use the following command to copy the Python application of a minimal TLS client, its private key, and the corresponding certificate to the STM32MP1 board. Also, the server certificate has to be placed on the board.

In [9]:
execute_vm_command('sshpass -p "root" scp tls_minimal/tls_client.py client.key client.crt server.crt root@192.168.13.13:/home/root/secure_communication')

### Run Server Application

Open a terminal and execute ``./tls_minimal/tls_server.py`` to start a simple Python-based TLS server.
If everything started correctly, you should see the prompt ``Waiting for client ...``.

In [11]:
execute_vm_command('gnome-terminal')

### Try to Run Client Application on STM32MP1 Board

Now, the client application on the STM32MP1 board can be started.

Open a terminal and connect to your board by SSH.

In [ ]:
execute_vm_command('gnome-terminal')

Then, navigate to the script's directory and execute it:

```bash
cd /home/root/secure_communication/
./tls_client.py

```

That does not seem to work...

What is the problem?

**Current TLS Connection Problem:**

ssl.SSLCertVerificationError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate is not yet valid (_ssl.c:997)


### Time and Date

Get the current time and date on the STM32MP1 board:

In [12]:
execute_stm32_command('date')
execute_stm32_command('/sbin/hwclock')

Thu Apr 28 23:32:10 UTC 2022

2000-01-01 00:15:56.784453+00:00



This data does not seem to be correct.

Use the tool ``date`` on your STM32MP1 board to set the correct time and date for today.

Example for Christmas 2000: ``date -s "Dec 24 2000 12:34:56"``

In [ ]:
execute_vm_command('gnome-terminal')

Check time and date again.

In [13]:
execute_stm32_command('date')
execute_stm32_command('/sbin/hwclock')

Thu Nov 26 12:19:15 UTC 2026

2000-01-01 00:17:57.972102+00:00



Did it work?

**Notes on Setting Time and Date:**

Need to check if the device has the actual time

### Start Server Again

Go back to the previously opened terminal and run the script ``tls_server.py`` again.

### Rerun TLS Client on STM32MP1 Board

Navigate to the script's directory on your board and execute it again:

```bash
cd /home/root/secure_communication/
./tls_client.py

```

Check, if the communication example ran as expected.

Which TLS version was negotiated and which cipher suite?

**Negotiated TLS Version:**

'TLSv1.3'

**Negotiated Cipher Suite:**

'TLS_AES_256_GCM_SHA384'

---

## Exercise 7.2: Using a Certificate Authority (CA)

The prior example used two completely unrelated certificates for the TLS communication. In this follow-up exercise, a self-signed CA certificate is used as the root for the server's and client's certificate chain.

### Create CA Key

First of all, a private RSA key is created for the CA.

In [14]:
execute_vm_command('openssl genrsa -out ca.key 2048')

Generating RSA private key, 2048 bit long modulus (2 primes)
........................................+++++
.......+++++
e is 65537 (0x010001)



### Generate CA Certificate

The self-signed CA root certificate is created by signing it with the same key.

In [15]:
execute_vm_command('openssl req -new -x509 -key ca.key -out ca.crt -days 1826 -config configs/ca.conf')

### Create Private Key and CSR for Client (STM32MP1 Board)

In the next steps, a key and a corresponding Certificate Signing Request (CSR) are generated for the client.

In [16]:
execute_vm_command('openssl genrsa -out stm32mp1.key 2048')
execute_vm_command('openssl req -new -key stm32mp1.key -out stm32mp1.csr -config configs/stm32mp1.conf -extensions "v3_req"')

Generating RSA private key, 2048 bit long modulus (2 primes)
.............................................................+++++
...................................................................................................................................................................................................+++++
e is 65537 (0x010001)




### Create Private Key and CSR for Server (VM)

The same has to be done for the server.

In [17]:
execute_vm_command('openssl genrsa -out vmserver.key 2048')
execute_vm_command('openssl req -new -key vmserver.key -out vmserver.csr -config configs/vmserver.conf -extensions "v3_req"')

Generating RSA private key, 2048 bit long modulus (2 primes)
....................................................................+++++
...........................+++++
e is 65537 (0x010001)




### Signing the CSRs by the CA

Finally, the CSRs can be signed by the CA, acting as a Trusted Third Party (TTP) in this scenario.

In [18]:
execute_vm_command('openssl x509 -req -in stm32mp1.csr -CA ca.crt -CAkey ca.key -CAcreateserial -extfile configs/stm32mp1.conf -extensions v3_req -out stm32mp1.crt -days 730')
execute_vm_command('openssl x509 -req -in vmserver.csr -CA ca.crt -CAkey ca.key -CAcreateserial -extfile configs/vmserver.conf  -extensions v3_req -out vmserver.crt -days 730')

Signature ok
subject=C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = STM32MP1
Getting CA Private Key

Signature ok
subject=C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = VM-Server
Getting CA Private Key



### Print Details of Created Certificates

The following commands print all information contained in the newly created certificates.

In [19]:
execute_vm_command('openssl x509 -in stm32mp1.crt -text -noout')

Certificate:
    Data:
        Version: 3 (0x2)
        Serial Number:
            5d:7e:9f:0f:ed:aa:b9:a7:94:dd:c1:41:80:16:8a:2a:e9:e0:c2:d1
        Signature Algorithm: sha256WithRSAEncryption
        Issuer: C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = EmbSec-Root-CA
        Validity
            Not Before: Nov 26 11:24:31 2026 GMT
            Not After : Nov 25 11:24:31 2028 GMT
        Subject: C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = STM32MP1
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                RSA Public-Key: (2048 bit)
                Modulus:
                    00:b0:d7:fe:c7:89:3c:6f:a3:f6:15:31:d8:ee:31:
                    5a:81:37:52:fd:02:63:79:a9:05:8f:41:61:e3:7a:
                    6b:79:a9:e1:4f:1f:d0:1e:09:79:6d:04:c0:09:92:
                    d7:1f:f4:45:88:75:6b:d1:7c:38:25:10:de:ee:2b:
                    25:08:2f:a7:d3:c0:a3:93:91:ab:13:40:30:47:a2:
                    1a:c2:5f:6a:ee:17:44:2b:38:d3:9

In [20]:
execute_vm_command('openssl x509 -in vmserver.crt -text -noout')

Certificate:
    Data:
        Version: 3 (0x2)
        Serial Number:
            5d:7e:9f:0f:ed:aa:b9:a7:94:dd:c1:41:80:16:8a:2a:e9:e0:c2:d2
        Signature Algorithm: sha256WithRSAEncryption
        Issuer: C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = EmbSec-Root-CA
        Validity
            Not Before: Nov 26 11:24:31 2026 GMT
            Not After : Nov 25 11:24:31 2028 GMT
        Subject: C = DE, O = HSA_innos, OU = EmbeddedSecurity, CN = VM-Server
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                RSA Public-Key: (2048 bit)
                Modulus:
                    00:cf:cd:fa:23:a4:70:dd:d2:a7:40:15:1c:91:bf:
                    2f:73:6d:d6:5c:04:43:a1:35:23:b8:e4:f2:1f:aa:
                    e3:f9:16:7b:96:07:a7:4e:e5:e2:dc:9d:58:59:64:
                    d4:c5:0e:e3:43:f3:d6:51:87:1e:3f:1e:b9:be:fd:
                    d6:52:77:b1:0a:a7:d7:8a:f4:76:7c:7f:a5:89:37:
                    ea:74:27:29:11:08:1c:62:ee:18:

Discuss the printed information of the two certificates!

Name at least two differences to the certificates generated in the previous exercise!

**Differences to Prior Certificates:**

The issuer and TLS Version

### Copy Client Data to STM32MP1 Board

On the STM32MP1 board, the client application, the CA certificate as well as the client key and certificate are required. The following command copies them.

In [21]:
execute_vm_command('sshpass -p "root" scp tls_with_ca/tls_ca_client.py ca.crt stm32mp1.key stm32mp1.crt root@192.168.13.13:/home/root/secure_communication')

### Start Server Application

Like in the previous example, a terminal has to be opened, in which the server application ``tls_with_ca/tls_ca_server.py`` must be started.

Again, the prompt ``Waiting for client ...`` should appear.

In [ ]:
execute_vm_command('gnome-terminal')

### Run Client Application

Navigate to the client's directory on your board and execute it again:

```bash
cd /home/root/secure_communication/
./tls_ca_client.py

```

Well done!

... but wait ...

There is something clearly wrong! Someone must have made a critical implementation error in the server application.

Find out what the problem is, which lines of code in the server Python script are responsible for that, and how to fix it!

**Critical Problem:**

ssl.SSLError: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:997)


**Responsible Lines of Code in ``tls_with_ca/tls_ca_server.py``:**

context.options ^= ssl.OP_NO_TLSv1_2
context.options ^= ssl.OP_NO_TLSv1_3

**Fix for the Problem:**

context.options = ssl.OP_NO_TLSv1_3
context.options = ssl.OP_NO_TLSv1_3